In [30]:
from gc import collect

import findspark
import pyspark
from pyspark.sql import SparkSession

In [31]:
"""
Exercise #39 - Order sensors by number of critical days
"""

'\nExercise #39 - Order sensors by number of critical days\n'

In [32]:
findspark.init()
sc = pyspark.SparkContext.getOrCreate()
spark = SparkSession.builder.getOrCreate()

In [33]:
"""
RDD SOLUTION
"""

'\nRDD SOLUTION\n'

In [47]:
inputRDD = sc.textFile("data/sensors.txt").map(lambda x: x.split(","))
inputRDD.collect()

[['s1', '2016-01-01', '20.5'],
 ['s2', '2016-01-01', '30.1'],
 ['s1', '2016-01-02', '60.2'],
 ['s2', '2016-01-02', '20.4'],
 ['s1', '2016-01-03', '55.5'],
 ['s2', '2016-01-03', '52.5']]

In [48]:
k = 1

In [49]:
filteredRDD = inputRDD.filter(lambda x: float(x[2]) > 50).map(lambda x: (x[0], 1))
filteredRDD.collect()

[('s1', 1), ('s1', 1), ('s2', 1)]

In [50]:
finalRDD = filteredRDD.reduceByKey(lambda x, y: x + y).sortBy(lambda x: x[1], ascending=False)
finalRDD.collect()

[('s1', 2), ('s2', 1)]

In [51]:
finalRDD.take(k)

[('s1', 2)]

In [52]:
"""
SPARKSQL SOLUTION
"""

'\nSPARKSQL SOLUTION\n'

In [53]:
inputDF = spark.read.load("data/sensors.txt", header=False, inferSchema=True, format="csv", sep=",")
inputDF.show()

+---+----------+----+
|_c0|       _c1| _c2|
+---+----------+----+
| s1|2016-01-01|20.5|
| s2|2016-01-01|30.1|
| s1|2016-01-02|60.2|
| s2|2016-01-02|20.4|
| s1|2016-01-03|55.5|
| s2|2016-01-03|52.5|
+---+----------+----+



In [54]:
inputDF.createOrReplaceTempView("sensors")


In [55]:
result = spark.sql("SELECT _c0, COUNT(_c2) FROM sensors "
                   "WHERE _c2 > 50 "
                   "GROUP BY _c0 "
                   "ORDER BY COUNT(_c2) DESC "
                   "LIMIT 1")
result.show()

+---+----------+
|_c0|count(_c2)|
+---+----------+
| s1|         2|
+---+----------+

